In [6]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [7]:
IMG_DIR = Path(r"C:\Users\jchmo\Desktop\archive (2)\images")

In [8]:
class PetsDataset(Dataset):
    def __init__(self, root, transform=None):
        self.files = sorted(root.glob("*.jpg"))
        # trieda = názov súboru bez posledného "_číslo"
        names = [f.stem.rsplit("_", 1)[0] for f in self.files]
        self.classes = sorted(set(names))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[n] for n in names]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")  # pár obrázkov je sivých/CMYK
        if self.transform:
            img = self.transform(img)
        return img, self.labels[i]

In [9]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(), norm,
])
eval_tf = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(), norm,
])

In [10]:
# dva datasety nad tými istými súbormi, aby validácia nemala augmentáciu
full_train = PetsDataset(IMG_DIR, train_tf)
full_eval = PetsDataset(IMG_DIR, eval_tf)
print(len(full_train), len(full_train.classes))  # ~7390, 37

7390 37


In [11]:
idx = list(range(len(full_train)))
train_idx, tmp_idx = train_test_split(
    idx, test_size=0.3, stratify=full_train.labels, random_state=42)
val_idx, test_idx = train_test_split(
    tmp_idx, test_size=0.5, stratify=[full_train.labels[i] for i in tmp_idx],
    random_state=42)

In [12]:
train_dl = DataLoader(Subset(full_train, train_idx), batch_size=32, shuffle=True, num_workers=0)
val_dl = DataLoader(Subset(full_eval, val_idx), batch_size=64, num_workers=0)
test_dl = DataLoader(Subset(full_eval, test_idx), batch_size=64, num_workers=0)